# Task 3 — Estimating magnet temperature in a real PMSM

**ELEC-E8131 · AI for Electrical Engineers · Colab lab, part 3 of 4**

### What you will learn about Colab
- storing credentials as **Colab Secrets** instead of pasting them into a cell
- downloading and caching a real dataset
- checkpointing, and resuming a run that was interrupted
- `DataLoader` and mini-batching
- running a sweep and logging results to Drive so they survive a restart
- when a GPU actually pays for itself, and how to benchmark it without fooling yourself

### What you will learn about machine learning
- **how to split data that is not independent** — the single most consequential decision here
- why you compute baselines before you build a model
- validation curves and early stopping
- what a hyperparameter sweep can and cannot tell you

---

### The engineering problem

In a permanent-magnet synchronous motor, the rotor magnets demagnetise irreversibly if they get too
hot. You cannot put a temperature sensor on a spinning rotor in a production vehicle, so the
temperature has to be *estimated* from quantities you can measure: currents, voltages, speed,
torque, coolant and ambient temperature.

This is a real, commercially important estimation problem. The data you will use is a real
measurement campaign: **185 hours of test-bench recordings from a 52 kW automotive traction PMSM**,
collected by the LEA department at Paderborn University and published on Kaggle. The rotor
temperature was obtained with an infrared telemetry unit — equipment you can afford on a test bench
and not in a car, which is precisely why a model is wanted.

The problem has one property that will dominate everything you do today:

> **Magnet temperature is a state, not a function of the present inputs.**

The magnet is hot because of the losses over the last several minutes, not because of the current
flowing right now. A motor at 200 A having just started from cold and the same motor at 200 A after
twenty minutes at full load look *identical* in the instantaneous measurements, and differ by tens
of kelvin in `pm`.

Keep that sentence in mind. Nearly every result below is a consequence of it.

---
## Part 1 — Get the data

You need a free Kaggle account and an API token.

1. Log in at kaggle.com, open **Settings → API → Create New Token**. A `kaggle.json` file downloads.
   It contains a username and a key.
2. In Colab, open the **key icon** in the left sidebar (Secrets).
3. Add two secrets: `KAGGLE_USERNAME` and `KAGGLE_KEY`, with the values from `kaggle.json`.
   Enable notebook access for both.

**Do not paste the key into a code cell.** Notebooks get shared, screenshotted and committed to
git. A credential in a cell is a credential you have published. Secrets are stored against your
Google account, are not part of the notebook file, and do not travel when you share it.

This is not a Colab quirk. It is the same reason you do not put passwords in source code anywhere.

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
print("credentials loaded for user:", os.environ["KAGGLE_USERNAME"])

In [ ]:
!pip install -q kagglehub

In [ ]:
import kagglehub

DATA_DIR = kagglehub.dataset_download("wkirgsn/electric-motor-temperature")
print("downloaded to:", DATA_DIR)
print("files:", os.listdir(DATA_DIR))

CSV_PATH = os.path.join(DATA_DIR, "measures_v2.csv")

`kagglehub` caches the download, so re-running the cell is cheap. The cache lives on the virtual
machine, though, so a runtime reset means downloading again.

**If the download fails**, the cause is almost always one of: the secrets are named wrongly (they
are case-sensitive), notebook access is not enabled for them, or you have not accepted the
dataset's terms on the Kaggle website. Open the dataset page in a browser once and the last one
resolves itself.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

df_raw = pd.read_csv(CSV_PATH)
print("raw:", df_raw.shape)
print(df_raw.columns.tolist())
print("\nmeasurement sessions:", df_raw["profile_id"].nunique())
print(df_raw.describe().T[["mean", "std", "min", "max"]].round(2))

### Which columns may we use?

The file contains four temperatures: `pm` (rotor magnets), and `stator_winding`, `stator_tooth`,
`stator_yoke`. In the original benchmark **all four are targets**, not inputs.

It is tempting to use the stator temperatures as features, and it would work spectacularly — they
are strongly coupled to `pm` through the same thermal path. It would also be meaningless, because
the whole point of the exercise is estimating a temperature you did not measure. Feeding one
unmeasured temperature in to predict another is not a solution to anything.

So the inputs are the eight quantities a drive controller actually has:

`ambient`, `coolant`, `u_d`, `u_q`, `motor_speed`, `i_d`, `i_q`, `torque`

**This is a modelling decision, not a data-cleaning step**, and it is the kind that quietly decides
whether a project is useful. If your features would not exist in the deployed system, your accuracy
number is fiction.

In [ ]:
FEATURES = ["ambient", "coolant", "u_d", "u_q", "motor_speed", "i_d", "i_q", "torque"]
TARGET = "pm"
EXCLUDED = ["stator_winding", "stator_tooth", "stator_yoke"]

# Recordings are at 2 Hz. The thermal dynamics have time constants of minutes, so the data is
# heavily oversampled for this task. Keep every 8th sample -> one row per 4 seconds.
DECIMATE = 8
DT = 0.5 * DECIMATE          # seconds per row after decimation

df = df_raw[df_raw.groupby("profile_id").cumcount() % DECIMATE == 0].reset_index(drop=True)

X = df[FEATURES].to_numpy(dtype=np.float64)
Y = df[TARGET].to_numpy(dtype=np.float64)
G = df["profile_id"].to_numpy()

print("after decimation: %d rows, %.1f hours of recording" % (len(df), len(df) * DT / 3600))
print("pm spans %.1f to %.1f degC, standard deviation %.2f K" % (Y.min(), Y.max(), Y.std()))

Decimating by 8 costs nothing here — a signal whose time constant is minutes does not need to be
sampled twice a second — and it makes everything below run in a lab session rather than overnight.

**But note what decimation does *not* fix.** Consecutive rows are now 4 seconds apart instead of
0.5 s. They are still, for thermal purposes, the same measurement. Keep that in mind for Part 3;
if you were hoping that thinning the data would make a random split acceptable, it does not.

---
## Part 2 — Look at the data, and compute baselines

The sessions vary from one to six hours, so they have very different lengths. Look at one.

In [ ]:
sizes = df.groupby("profile_id").size().sort_values(ascending=False)
print("rows per session (longest 10):")
print(sizes.head(10))
print("\nshortest:", sizes.min(), " longest:", sizes.max())

pid = sizes.index[0]
one = df[df.profile_id == pid]
tmin = np.arange(len(one)) * DT / 60.0

fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
ax[0].plot(tmin, one["motor_speed"], lw=.8); ax[0].set_ylabel("speed [rpm]")
ax[1].plot(tmin, one["torque"], lw=.8, color="tab:orange"); ax[1].set_ylabel("torque [Nm]")
ax[2].plot(tmin, one["coolant"], lw=1.2, label="coolant")
ax[2].plot(tmin, one["pm"], lw=1.2, label="magnet (pm)")
ax[2].set_ylabel("temperature [degC]"); ax[2].set_xlabel("time [min]"); ax[2].legend()
for a in ax: a.grid(alpha=.3)
plt.suptitle("session %s" % pid); plt.tight_layout(); plt.show()

**Look at the lag.** Speed and torque swing around on a timescale of seconds. `pm` drifts up and
down over many minutes and is still moving long after the load has changed. The magnet temperature
at any instant encodes several minutes of history that appears nowhere in the row of the table you
are about to feed a network.

Now measure how similar neighbouring rows are.

In [ ]:
step = np.abs(np.diff(one["pm"].to_numpy()))
print("typical change in pm between consecutive rows (%.0f s apart): %.4f K" % (DT, np.median(step)))
print("standard deviation of pm across the whole dataset:            %.2f K" % Y.std())
print("ratio: %.0f x" % (Y.std() / max(np.median(step), 1e-9)))

Consecutive rows differ by a tiny fraction of the overall spread. Row *k* and row *k+1* are, for
all practical purposes, **the same data point measured twice.**

Hold on to that ratio. It is the entire content of Part 3.

### Splitting by session

Sessions have very different lengths, so we cannot just count them off. Shuffle the session IDs
with a fixed seed and assign whole sessions to train, validation and test until each has roughly
its share of the rows.

In [ ]:
rs = np.random.default_rng(0)
pids = sizes.index.to_numpy().copy()
rs.shuffle(pids)

target_tr, target_va = 0.70 * len(df), 0.15 * len(df)
tr_p, va_p, te_p, running = [], [], [], 0
for p in pids:
    n = sizes[p]
    if running < target_tr:
        tr_p.append(p)
    elif running < target_tr + target_va:
        va_p.append(p)
    else:
        te_p.append(p)
    running += n

tr_idx = np.where(np.isin(G, tr_p))[0]
va_idx = np.where(np.isin(G, va_p))[0]
te_idx = np.where(np.isin(G, te_p))[0]

print("train %6d rows, %2d sessions" % (len(tr_idx), len(tr_p)))
print("val   %6d rows, %2d sessions" % (len(va_idx), len(va_p)))
print("test  %6d rows, %2d sessions" % (len(te_idx), len(te_p)))

### Baselines

A model is only impressive relative to the trivial thing you did not bother to build. Compute three
before training anything:

1. **Predict the training mean.** The floor. Beating this is not an achievement.
2. **Predict the coolant temperature.** A real engineer's first guess, requiring no fitting at all.
3. **Linear regression on the eight features.** The simplest fitted model.

In [ ]:
rmse = lambda a, b: float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

mu, sd = X[tr_idx].mean(axis=0), X[tr_idx].std(axis=0)
Xs = (X - mu) / sd
Ad = lambda i: np.column_stack([Xs[i], np.ones(len(i))])
coef, *_ = np.linalg.lstsq(Ad(tr_idx), Y[tr_idx], rcond=None)

BASE_MEAN = rmse(Y[tr_idx].mean(), Y[va_idx])
BASE_COOL = rmse(df["coolant"].to_numpy()[va_idx], Y[va_idx])
BASE_LIN = rmse(Ad(va_idx) @ coef, Y[va_idx])

print("validation RMSE [K]")
print("  predict training mean : %6.2f" % BASE_MEAN)
print("  predict coolant temp  : %6.2f" % BASE_COOL)
print("  linear regression     : %6.2f" % BASE_LIN)

Write these three numbers down. **A neural network that does not clearly beat all three has earned
nothing**, and you will refer back to them repeatedly.

Pay particular attention to the coolant baseline. In steady state the magnet sits at roughly the
coolant temperature plus a load-dependent offset, so "report the coolant reading" is a genuinely
competitive estimator and costs nothing to implement.

---
## Part 3 — The split that lies to you

Here is the most important part of this lab.

We will train exactly the same network twice. The only difference is **how the rows are divided
into training and test sets**. Same architecture, same optimiser, same epochs, same seed.

In [ ]:
def fit_mlp(train_idx, eval_idx, Xa=None, width=64, depth=3, lr=1e-3,
            epochs=20, batch_size=1024, seed=0):
    Xa = X if Xa is None else Xa
    # Scaling constants come from the TRAINING rows only, always.
    m_, s_ = Xa[train_idx].mean(axis=0), Xa[train_idx].std(axis=0)
    s_ = np.where(s_ < 1e-9, 1.0, s_)
    ym, ys = Y[train_idx].mean(), Y[train_idx].std()
    Xn, Yn = (Xa - m_) / s_, (Y - ym) / ys

    T = lambda i, a: torch.tensor(a[i], dtype=torch.float32)
    xt, yt = T(train_idx, Xn), T(train_idx, Yn).view(-1, 1)
    xe, ye = T(eval_idx, Xn), T(eval_idx, Yn).view(-1, 1)

    torch.manual_seed(seed)
    layers, d = [], Xa.shape[1]
    for _ in range(depth):
        layers += [nn.Linear(d, width), nn.ReLU()]; d = width
    model = nn.Sequential(*layers, nn.Linear(d, 1))
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    n = len(train_idx)
    for epoch in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            b = perm[i:i + batch_size]
            opt.zero_grad()
            ((model(xt[b]) - yt[b]) ** 2).mean().backward()
            opt.step()

    with torch.no_grad():
        tr_rmse = torch.sqrt(((model(xt) - yt) ** 2).mean()).item() * ys
        ev_rmse = torch.sqrt(((model(xe) - ye) ** 2).mean()).item() * ys
    return model, tr_rmse, ev_rmse


def make_predictor(model, train_idx, Xa=None):
    # Reproduces exactly the scaling fit_mlp used, and undoes it on the way out.
    Xa = X if Xa is None else Xa
    m_, s_ = Xa[train_idx].mean(axis=0), Xa[train_idx].std(axis=0)
    s_ = np.where(s_ < 1e-9, 1.0, s_)
    ym_, ys_ = Y[train_idx].mean(), Y[train_idx].std()

    def predict(rows):
        with torch.no_grad():
            z = torch.tensor((Xa[rows] - m_) / s_, dtype=torch.float32)
            return model(z).numpy().ravel() * ys_ + ym_
    return predict

In [ ]:
%%time
# SPLIT A: shuffle every row and take 75 % / 25 %, ignoring which session it came from.
r = np.random.default_rng(0)
shuffled = r.permutation(len(Y))
cut = int(0.75 * len(Y))
rand_tr, rand_te = shuffled[:cut], shuffled[cut:]

_, tr_a, te_a = fit_mlp(rand_tr, rand_te)
print("RANDOM ROW SPLIT   train RMSE %.2f K | test RMSE %.2f K" % (tr_a, te_a))
print("coolant baseline on the same rows: %.2f K"
      % rmse(df["coolant"].to_numpy()[rand_te], Y[rand_te]))

A striking result on held-out data — far better than any baseline.

If you got this in a project meeting you would present it. Take a moment to believe it, because you
are about to find out it is worthless.

In [ ]:
%%time
# SPLIT B: hold out entire measurement sessions. Nothing from a test session is ever seen.
group_tr = np.concatenate([tr_idx, va_idx])
_, tr_b, te_b = fit_mlp(group_tr, te_idx)
print("GROUPED SPLIT      train RMSE %.2f K | test RMSE %.2f K" % (tr_b, te_b))
print()
print("random-split test error : %.2f K" % te_a)
print("grouped-split test error: %.2f K" % te_b)
print("the same model is %.1fx worse when tested honestly" % (te_b / te_a))

### What happened

Nothing about the model changed. What changed is whether the test set contained information that
was also in the training set.

Recall the ratio from Part 2. Under a random split, for a typical test row, rows *k−1* and *k+1*
are almost certainly in the **training** set, with nearly identical feature values and nearly
identical `pm`. The network does not have to model thermal physics at all. It only has to
interpolate between two neighbours it has memorised.

The grouped split removes that crutch. Now the model must predict `pm` for a bench session whose
thermal history it has never seen — and the instantaneous features do not determine `pm`, exactly
as the introduction warned.

**The first figure was not a mistake in the code. It was a correct answer to the wrong question.**

### The rule

> Split along the axis of independence in your data, not along the row index.

This is not a special case. It recurs constantly:
- **time series** — split by time, never randomly, or you train on the future;
- **medical data** — split by patient, not by scan;
- **speech** — split by speaker;
- **industrial data** — split by machine, batch, or measurement session.

The tell is always the same question: *could two rows in different splits be near-duplicates of
each other?* If yes, the split is wrong.

Note that in Task 1 a random split **was** correct, because each measurement was an independent
noisy draw. The rule is about the data, not about a preference for one splitting function.

### Now try this

Increase `epochs` in the random-split call and watch the test error fall further. The more you
train, the better the memorisation gets, and the more impressive the meaningless number becomes.

Then try decimating harder — set `DECIMATE = 64` at the top and re-run. Rows are then over half a
minute apart. Does the random split become honest? (It does not. Thinning correlated data does not
make it independent; it only makes it smaller.)

No amount of careful modelling rescues you from a bad split, and no diagnostic inside the training
loop warns you. Only knowing your data does.

---
## Part 4 — Validation curves, early stopping, checkpointing

From here on we use the honest three-way session split from Part 2.

We also switch to PyTorch's `DataLoader` rather than hand-written batching. It handles shuffling
and batching, and it is what you will see in every codebase you inherit.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

m_, s_ = X[tr_idx].mean(axis=0), X[tr_idx].std(axis=0)
s_ = np.where(s_ < 1e-9, 1.0, s_)
ym, ys = Y[tr_idx].mean(), Y[tr_idx].std()
Xn, Yn = (X - m_) / s_, (Y - ym) / ys

T = lambda i, a: torch.tensor(a[i], dtype=torch.float32)
x_tr, y_tr = T(tr_idx, Xn), T(tr_idx, Yn).view(-1, 1)
x_va, y_va = T(va_idx, Xn), T(va_idx, Yn).view(-1, 1)
x_te, y_te = T(te_idx, Xn), T(te_idx, Yn).view(-1, 1)

train_loader = DataLoader(TensorDataset(x_tr, y_tr), batch_size=1024, shuffle=True)
print("%d batches per epoch" % len(train_loader))

In [ ]:
import time

CKPT_DIR = "/content/checkpoints"        # deliberately NOT on Drive - see below
os.makedirs(CKPT_DIR, exist_ok=True)


def train_with_validation(width=128, depth=2, lr=1e-2, epochs=40, seed=0,
                          ckpt_path=None):
    ckpt_path = ckpt_path or os.path.join(CKPT_DIR, "best.pt")
    torch.manual_seed(seed)
    layers, d = [], x_tr.shape[1]
    for _ in range(depth):
        layers += [nn.Linear(d, width), nn.ReLU()]; d = width
    model = nn.Sequential(*layers, nn.Linear(d, 1))
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    history, best_val, best_epoch = [], np.inf, -1
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            ((model(xb) - yb) ** 2).mean().backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            tr_r = torch.sqrt(((model(x_tr) - y_tr) ** 2).mean()).item() * ys
            va_r = torch.sqrt(((model(x_va) - y_va) ** 2).mean()).item() * ys
        history.append((epoch, tr_r, va_r))

        if va_r < best_val:
            best_val, best_epoch = va_r, epoch
            torch.save({"state_dict": model.state_dict(), "epoch": epoch,
                        "val_rmse": va_r, "width": width, "depth": depth},
                       ckpt_path)
    return model, np.array(history), best_val, best_epoch

In [ ]:
%%time
model, hist, best_val, best_epoch = train_with_validation()

print("best validation RMSE %.2f K at epoch %d" % (best_val, best_epoch))
print("validation RMSE at the LAST epoch: %.2f K" % hist[-1, 2])
print("training RMSE at the last epoch  : %.2f K" % hist[-1, 1])

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(hist[:, 0], hist[:, 1], label="train")
plt.plot(hist[:, 0], hist[:, 2], label="validation")
plt.axvline(best_epoch, ls="--", color="k", lw=1, label="best epoch")
plt.axhline(BASE_COOL, ls=":", color="tab:red", label="coolant baseline")
plt.xlabel("epoch"); plt.ylabel("RMSE [K]"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

Two things to read off this plot.

**Training error keeps falling; validation error does not.** Past the marked epoch, every further
epoch makes the model *worse* at the job you care about. Stopping at the best validation epoch —
**early stopping** — is the cheapest regulariser in existence, and the checkpointing code above has
already saved that model for you.

**Compare against the red line.** How much of the gap between the coolant baseline and zero has the
network actually closed? Be honest about the answer; we return to it in Part 7.

### About where the checkpoints went

`CKPT_DIR` is `/content/checkpoints`, on the virtual machine. Writing a checkpoint every epoch to
Drive would be slow — Drive is a network filesystem, far slower than local disk.

The usual pattern: checkpoint frequently to local disk, copy the *best* one to Drive occasionally.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import shutil

DRIVE_OUT = "/content/drive/MyDrive/elec_e8131_colab/task3"
os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copy(os.path.join(CKPT_DIR, "best.pt"), os.path.join(DRIVE_OUT, "best.pt"))

# Save the preprocessing constants too. Weights alone are not a model.
np.savez(os.path.join(DRIVE_OUT, "scaler.npz"),
         mu=m_, sd=s_, y_mean=ym, y_std=ys, features=np.array(FEATURES))
print("copied to Drive:", os.listdir(DRIVE_OUT))

### Now try this

Restart the runtime (**Runtime → Restart session**), then run only the imports and this line:

```python
ck = torch.load("/content/checkpoints/best.pt")
```

It fails: `/content` was wiped, and so was the Kaggle download cache. Now try the Drive copy
instead. This is the difference between an overnight run you can recover and one you have to
repeat.

---
## Part 5 — Hyperparameter sweep

You have used `width=128, depth=2, lr=1e-2` because they were written in the notebook. That is not
a justification. Let us find out whether they matter.

Grid: **depth** {1, 2, 4} × **width** {16, 128} × **learning rate** {1e-2, 1e-3, 1e-4} = 18 runs.

To keep this to a few minutes we screen on a **random subsample of the training rows**. That is
normal practice — a sweep is for finding the right region, not the final model — but it is a real
approximation, and the ranking it produces can shift when you retrain on everything. Say so when
you report sweep results.

We record the **best** validation RMSE reached during each run, not the final one, because with
early stopping in place the best is what you would deploy.

In [ ]:
sub = np.random.default_rng(1).permutation(len(tr_idx))[:40000]
x_sw, y_sw = x_tr[sub], y_tr[sub]
sweep_loader = DataLoader(TensorDataset(x_sw, y_sw), batch_size=1024, shuffle=True)
print("sweeping on %d of %d training rows" % (len(sub), len(tr_idx)))


def sweep_run(width, depth, lr, epochs=15, seed=0):
    torch.manual_seed(seed)
    layers, d = [], x_tr.shape[1]
    for _ in range(depth):
        layers += [nn.Linear(d, width), nn.ReLU()]; d = width
    model = nn.Sequential(*layers, nn.Linear(d, 1))
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    best, final = np.inf, np.inf
    for epoch in range(epochs):
        for xb, yb in sweep_loader:
            opt.zero_grad()
            ((model(xb) - yb) ** 2).mean().backward()
            opt.step()
        with torch.no_grad():
            v = torch.sqrt(((model(x_va) - y_va) ** 2).mean()).item() * ys
        best, final = min(best, v), v
    return best, final

In [ ]:
%%time
rows = []
for depth in (1, 2, 4):
    for width in (16, 128):
        for lr in (1e-2, 1e-3, 1e-4):
            best, final = sweep_run(width, depth, lr)
            rows.append(dict(depth=depth, width=width, lr=lr,
                             best_val=best, final_val=final))
            print("depth %d  width %3d  lr %.0e   best %.2f K   final %.2f K"
                  % (depth, width, lr, best, final))

sweep = pd.DataFrame(rows)

In [ ]:
sweep.to_csv(os.path.join(DRIVE_OUT, "sweep_results.csv"), index=False)
print("saved - safe from a runtime reset")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, w in zip(axes, (16, 128)):
    piv = sweep[sweep.width == w].pivot(index="depth", columns="lr", values="best_val")
    im = ax.imshow(piv.values, cmap="viridis_r")
    ax.set_xticks(range(len(piv.columns)), ["%.0e" % c for c in piv.columns])
    ax.set_yticks(range(len(piv.index)), piv.index)
    ax.set_xlabel("learning rate"); ax.set_ylabel("depth"); ax.set_title("width %d" % w)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            ax.text(j, i, "%.2f" % piv.values[i, j], ha="center", va="center", color="w")
    fig.colorbar(im, ax=ax, label="best val RMSE [K]")
plt.tight_layout(); plt.show()

print("best configuration:")
print(sweep.loc[sweep.best_val.idxmin()])
print("\nspread across the grid: %.2f to %.2f K" % (sweep.best_val.min(), sweep.best_val.max()))
print("coolant baseline        : %.2f K" % BASE_COOL)
print("linear regression       : %.2f K" % BASE_LIN)

### Read this result honestly

Three observations, in increasing order of importance.

**1. Most of the grid is bunched together.** Set aside the configurations that plainly have not
finished training (low learning rate, few epochs) and the rest lands in a narrow band. A factor of
eight in width and a factor of four in depth barely move it.

**2. Early stopping matters more than architecture.** Compare the `best` and `final` columns. For
the larger models at high learning rate the two differ by more than the spread across the usable
part of the grid. *When* you stopped has a bigger effect than every architectural choice you made.
This also explains the width-512 result from Task 1, Part 6.

**3. Compare against the baselines.** After eighteen neural networks, how far ahead of linear
regression are you? And of simply reporting the coolant temperature?

That third point is not a failure of your code. It is the answer to a question worth asking, and
the sweep is how you found it out. A sweep tells you whether you are on the right plateau. Here it
should tell you clearly: **you are not going to fix this by tuning.** The bottleneck is elsewhere.

You already know where. Go back and re-read the sentence in bold at the top of this notebook.

---
## Part 6 — Now the GPU earns its keep

In Task 1 the GPU was *slower*, because each kernel launch did almost no arithmetic. Here you have
a hundred thousand training rows, batches of 1024, and up to 128-unit layers. The arithmetic per
launch is orders of magnitude larger.

Switch to a GPU runtime (**Runtime → Change runtime type → T4 GPU**) and re-run the notebook up to
this point, then run the cell below.

It measures **three** things, and the comparison between the second and third is the real lesson.

In [ ]:
def bench(device, mode, width=128, depth=4, epochs=3, batch_size=1024):
    torch.manual_seed(0)
    layers, d = [], x_tr.shape[1]
    for _ in range(depth):
        layers += [nn.Linear(d, width), nn.ReLU()]; d = width
    model = nn.Sequential(*layers, nn.Linear(d, 1)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    if mode == "loader":
        source = DataLoader(TensorDataset(x_tr, y_tr), batch_size=batch_size, shuffle=True)
    else:                                   # keep the whole dataset on the device
        xd, yd = x_tr.to(device), y_tr.to(device)

    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(epochs):
        if mode == "loader":
            for xb, yb in source:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad(); ((model(xb) - yb) ** 2).mean().backward(); opt.step()
        else:
            perm = torch.randperm(len(xd), device=device)
            for i in range(0, len(xd), batch_size):
                b = perm[i:i + batch_size]
                opt.zero_grad(); ((model(xd[b]) - yd[b]) ** 2).mean().backward(); opt.step()
    if device == "cuda":
        torch.cuda.synchronize()
    return time.time() - t0


t_cpu = bench("cpu", "loader")
print("CPU, DataLoader          : %6.2f s" % t_cpu)
if torch.cuda.is_available():
    t_gpu_loader = bench("cuda", "loader")
    t_gpu_res = bench("cuda", "resident")
    print("GPU, DataLoader          : %6.2f s   (%.2fx vs CPU)" % (t_gpu_loader, t_cpu / t_gpu_loader))
    print("GPU, data already on GPU : %6.2f s   (%.2fx vs CPU)" % (t_gpu_res, t_cpu / t_gpu_res))
else:
    print("No GPU on this runtime - switch it on and re-run.")

### What the three numbers mean

Record all three and interpret them, rather than reporting only the best.

- **GPU/DataLoader barely beats CPU.** Then your bottleneck is the data pipeline, not the model.
  Each batch is assembled in Python and copied across the PCIe bus; the GPU waits. This is the most
  common reason a GPU "doesn't help", and buying a faster one would change nothing.
- **GPU/resident is much faster than GPU/DataLoader.** Confirms the above: same arithmetic, same
  device, only the data path changed.
- **Both GPU numbers beat CPU comfortably.** Then you are compute-bound and the GPU is doing what
  it is for.

The general lesson is about benchmarking, not hardware: **a timing number is only as good as your
understanding of what it includes.** In Task 1 you concluded a GPU can be slower. Here you can see
the same GPU being slower or faster than itself, depending on how you feed it.

For a dataset that fits comfortably in GPU memory — as this one does — keeping it resident is
simply the right thing to do. `DataLoader` earns its complexity when data does not fit, or when
loading involves real work like decoding images.

---
## Part 7 — The fix that actually works

The sweep said tuning will not save you. The introduction said why: `pm` depends on the *history*
of the losses, and no row of the table contains any history.

So put the history in the table.

For each session, compute exponentially weighted moving averages of the copper-loss proxy
$i_d^2 + i_q^2$ over three different horizons, plus a slow average of the coolant temperature.
These are cheap causal filters — a microcontroller computes each one with a multiply-add and no
buffer, and they use no future information, which a real-time estimator is not allowed to do.

**Compute them per session.** An EWMA that ran across a session boundary would blend one bench test
into the next, which is a leak of exactly the kind Part 3 was about. `groupby().transform()` does
this correctly; a plain `.ewm()` on the whole column would not.

This is not an invention of ours: EWMA features of the input signals are the standard approach in
the literature on this dataset, for precisely the reason above.

In [ ]:
loss_proxy = df["i_d"] ** 2 + df["i_q"] ** 2
work = pd.DataFrame({"profile_id": df["profile_id"],
                     "loss": loss_proxy,
                     "coolant": df["coolant"]})

HORIZONS_S = [120, 600, 3000]        # about 2 min, 10 min, 50 min of memory
hist_cols, HIST_NAMES = [], []

for h in HORIZONS_S:
    span = max(2.0, h / DT)          # span in samples
    col = work.groupby("profile_id")["loss"].transform(
        lambda s: s.ewm(span=span, adjust=False).mean())
    hist_cols.append(col.to_numpy()); HIST_NAMES.append("loss_ewma_%ds" % h)

col = work.groupby("profile_id")["coolant"].transform(
    lambda s: s.ewm(span=max(2.0, 600 / DT), adjust=False).mean())
hist_cols.append(col.to_numpy()); HIST_NAMES.append("coolant_ewma_600s")

X_hist = np.column_stack([X] + hist_cols)
print("features: %d -> %d" % (X.shape[1], X_hist.shape[1]))
print("added:", HIST_NAMES)

In [ ]:
%%time
_, tr_0, va_0 = fit_mlp(tr_idx, va_idx, Xa=X,      width=128, depth=2, lr=1e-3, epochs=20)
_, tr_h, va_h = fit_mlp(tr_idx, va_idx, Xa=X_hist, width=128, depth=2, lr=1e-3, epochs=20)

print("8 instantaneous features   : validation RMSE %.2f K" % va_0)
print("12 features with history   : validation RMSE %.2f K" % va_h)
print("best of the 18-run sweep   : validation RMSE %.2f K" % sweep.best_val.min())
print("linear regression baseline : validation RMSE %.2f K" % BASE_LIN)
print("coolant baseline           : validation RMSE %.2f K" % BASE_COOL)

Four extra columns, computed with a groupby and an `ewm`, against eighteen tuned architectures.
Compare the improvement each bought.

**This is the third time in this lab that the same thing has happened.**

| task | what was tried | what actually worked |
|------|----------------|----------------------|
| 1 | wider and deeper networks | taking $\log_{10}$ of the target |
| 2 | reweighting the loss | adding a slew-rate feature |
| 3 | 18-configuration sweep | four EWMA columns |

In all three, the model class was never the constraint. The **representation of the problem** was.
Architecture search is what you do *after* establishing that your inputs contain the information
the task requires — and establishing that is engineering judgement about the system, not something
a sweep can discover for you.

This is also why domain knowledge is not a nice-to-have in engineering machine learning. Knowing
that a magnet is a thermal mass with a time constant of minutes is what produced those four
columns. No amount of hyperparameter search would have.

---
## Part 8 — Final evaluation

You have made every decision on validation data. Touch the test set once.

In [ ]:
%%time
final_model, tr_f, te_f = fit_mlp(np.concatenate([tr_idx, va_idx]), te_idx, Xa=X_hist,
                                  width=128, depth=2, lr=1e-3, epochs=20)
print("TEST RMSE: %.2f K   (train %.2f K)" % (te_f, tr_f))
print()
print("on the same test sessions:")
print("  predict training mean : %.2f K" % rmse(Y[tr_idx].mean(), Y[te_idx]))
print("  predict coolant temp  : %.2f K" % rmse(df["coolant"].to_numpy()[te_idx], Y[te_idx]))
print("  random-split fantasy  : %.2f K" % te_a)

In [ ]:
# Per-session errors: an average hides a lot.
fit_rows = np.concatenate([tr_idx, va_idx])
predict = make_predictor(final_model, fit_rows, Xa=X_hist)

per_session = []
for p in te_p:
    m = np.where(G == p)[0]
    yh = predict(m)
    per_session.append(dict(profile_id=p, rows=len(m),
                            rmse_K=rmse(yh, Y[m]),
                            max_abs_err_K=float(np.max(np.abs(yh - Y[m])))))

per_session = pd.DataFrame(per_session).sort_values("rmse_K", ascending=False)
print(per_session.round(2).to_string(index=False))
print("\nworst single-sample error anywhere in the test set: %.1f K"
      % per_session.max_abs_err_K.max())

### Read the final number honestly

Compare your test RMSE against the coolant baseline on the same sessions, not against zero. Then
look at the per-session table — and in particular at the worst-case column.

Two reactions are both wrong.

It is **not** a failure if the margin over the baseline is modest. The improvement is real, it is
measured on sessions the model never saw, and this pipeline is a simplified version of how the
problem is actually solved in industry.

But it is **not** a success to be oversold either. If you presented a single RMSE without the
coolant baseline beside it, your audience would have no way to know how much of that performance
was available for free. Reporting a metric without its baseline is the most common way engineers
mislead themselves and everyone reading their slides.

And for this application the mean is the wrong summary anyway. Demagnetisation is a **threshold**
phenomenon: what matters is the largest error on the worst session, not the average error over all
of them. An estimator with an excellent RMSE and one 30 K excursion is not a protection function.

---
## Hand-in

Share the notebook (**Share → Anyone with the link → Viewer**) with your answers in text cells.

1. In one sentence, state the property of this dataset that makes a random row split invalid, and
   give one other domain where the same property appears.
2. Report your two numbers from Part 3. **Neither is a bug.** Explain precisely what question each
   one answers.
3. From the sweep: which mattered more, the architecture or the stopping epoch? Support your answer
   with two numbers from your own results table.
4. Report all three timings from Part 6 and say what they tell you about where the time went. If
   your GPU/DataLoader time was close to your CPU time, explain why a faster GPU would not have
   helped.
5. Explain why the EWMA features had to be computed per session. What would the validation RMSE
   have looked like if you had computed them across the whole concatenated file, and would that
   number have been trustworthy?
6. We excluded `stator_winding`, `stator_tooth` and `stator_yoke` from the features. Argue for or
   against that decision. Under what deployment assumption would including them be legitimate?
7. Using your per-session table, would you deploy this model as a demagnetisation protection
   function? Answer as an engineer: state what additional evidence you would want.

### Optional extensions

- **Do the sweep properly.** Add the EWMA features and re-run the grid on the full training set.
  Does the ranking change once the inputs are informative? The first sweep searched the wrong axis.
- **More horizons.** The published work on this dataset uses several EWMA spans on *every* input
  signal, not just two of them. How far does that get you, and where does it stop paying?
- **Predict the derivative.** Train on the change in `pm` per step and integrate. This is how a real
  observer is structured, and it makes the state explicit rather than hoping the network infers it.
- **How few sessions do you need?** Retrain on 4, 8, 16, all training sessions and plot validation
  RMSE against the count. Note that the relevant unit is *sessions*, not rows — the same insight as
  Part 3, in a different costume.